<a href="https://colab.research.google.com/github/kaouchounesalah-eddine-ux/arabic-news-classification/blob/main/04_multilingual_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Load dataset & Keep final test set

In [24]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import pandas as pd

from sklearn.model_selection import train_test_split
# from preprocess import preprocess


# =========================
# 1. Load dataset
# =========================

import pandas as pd

df = pd.read_json(
    '/content/drive/MyDrive/articles.json'
)

X = df["body"]
y = df["categories"].str[0]


# =========================
# 2. Keep final test set
# =========================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


2. Development Data

In [26]:
print("=== DEVELOPMENT DATA ===")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("\nNumber of samples:", len(X_train))

print("\nCategory distribution:")
print(y_train.value_counts())

=== DEVELOPMENT DATA ===
X_train shape: (16800,)
y_train shape: (16800,)

Number of samples: 16800

Category distribution:
categories
ثقافة     2800
دولي      2800
اقتصاد    2800
رياضة     2800
سياسة     2800
مجتمع     2800
Name: count, dtype: int64


3. Preprocessing

In [27]:
import unicodedata


def normalize_arabic(text):
    return (
        text.replace("أ", "ا")
            .replace("إ", "ا")
            .replace("آ", "ا")
    )


def remove_diacritics(text):
    return "".join(
        char
        for char in text
        if unicodedata.category(char) != "Mn"

    )


def remove_tatweel(text):
    return text.replace("ـ", "")

def remove_taarif(text):
    return " ".join(
        word.removeprefix("ال")
        for word in text.split()
    )

def remove_numbers(text):
    return "".join(
        char
        for char in text
        if not char.isdigit()
    )

def remove_marks(text):
    return text.replace('\\"', '"').replace('"', '') if isinstance(text, str) else text

arabic_stopwords = {
    "في", "من", "إلى", "على", "عن", "و", "أو",
    "أن", "إن", "كان", "كانت", "هذا", "هذه",
    "ذلك", "التي", "الذي", "هو", "هي", "هم",
    "ما", "لا", "لم", "لن", "مع", "كما"
}

def remove_stopwords(text):
    return " ".join(
        word
        for word in text.split()
        if word not in arabic_stopwords
    )

def preprocess(text):
    text = normalize_arabic(text)
    text = remove_diacritics(text)
    text = remove_tatweel(text)
    text = remove_taarif(text)
    text = remove_numbers(text)
    text = remove_marks(text)
    text = remove_stopwords(text)

    return text

In [28]:
X_train = X_train.apply(preprocess)
X_test = X_test.apply(preprocess)

4. X_train Embedding

In [29]:
from sentence_transformers import SentenceTransformer

arabic_embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [30]:
print("multilingual embedding model loaded.")
test_embeddings_ar = arabic_embedding_model.encode(
    X_train.iloc[:5].tolist()
)

print("Embeddings shape:", test_embeddings_ar.shape)

multilingual embedding model loaded.
Embeddings shape: (5, 384)


In [31]:
X_train_embeddings_ar = arabic_embedding_model.encode(
    X_train.tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("Embeddings shape:", X_train_embeddings_ar.shape)

Batches:   0%|          | 0/525 [00:00<?, ?it/s]

Embeddings shape: (16800, 384)


In [32]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_scores_ar = []

for fold, (train_idx, val_idx) in enumerate(
    skf.split(X_train_embeddings_ar, y_train),
    start=1
):

    X_fold_train = X_train_embeddings_ar[train_idx]
    X_fold_val = X_train_embeddings_ar[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    model = LogisticRegression(
        C=1.0,
        max_iter=5000
    )

    model.fit(X_fold_train, y_fold_train)

    y_fold_pred = model.predict(X_fold_val)

    fold_f1 = f1_score(
        y_fold_val,
        y_fold_pred,
        average="macro"
    )

    fold_scores_ar.append(fold_f1)

    print(
        f"Fold {fold}: "
        f"Macro F1 = {fold_f1:.4f}"
    )



Fold 1: Macro F1 = 0.8266
Fold 2: Macro F1 = 0.8326
Fold 3: Macro F1 = 0.8402
Fold 4: Macro F1 = 0.8304
Fold 5: Macro F1 = 0.8343


In [33]:
print("\nmultilingual Embedding + Logistic Regression")

print(
    f"Mean Macro F1: "
    f"{np.mean(fold_scores_ar):.4f}"
)

print(
    f"Standard deviation: "
    f"{np.std(fold_scores_ar):.4f}"
)


multilingual Embedding + Logistic Regression
Mean Macro F1: 0.8328
Standard deviation: 0.0045


 X_test embeddings

In [34]:
X_test_embeddings_ar = arabic_embedding_model.encode(
    X_test.tolist(),
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/132 [00:00<?, ?it/s]

Final test

In [35]:

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# 1. Train on ALL training embeddings
final_model_ar = LogisticRegression(
    C=1.0,
    max_iter=5000
)

final_model_ar.fit(
    X_train_embeddings_ar,
    y_train
)

print("Final model trained!")

# 2. Predict the held-out test set
y_test_pred_ar = final_model_ar.predict(
    X_test_embeddings_ar
)

# 3. Calculate evaluation metrics
accuracy = accuracy_score(
    y_test,
    y_test_pred_ar
)

macro_f1 = f1_score(
    y_test,
    y_test_pred_ar,
    average="macro"
)

print("\n===== FINAL TEST RESULTS =====")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")

# 4. Show precision, recall, and F1 for each category
print("\n===== CLASSIFICATION REPORT =====")
print(
    classification_report(
        y_test,
        y_test_pred_ar,
        zero_division=0
    )
)

# 5. Confusion matrix
labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    y_test_pred_ar,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=labels,
    columns=labels
)

print("\n===== CONFUSION MATRIX =====")
print("Rows = actual labels; columns = predicted labels")
display(cm_df)

Final model trained!

===== FINAL TEST RESULTS =====
Accuracy: 0.8402
Macro F1: 0.8402

===== CLASSIFICATION REPORT =====
              precision    recall  f1-score   support

      اقتصاد       0.82      0.79      0.80       700
       ثقافة       0.90      0.91      0.90       700
        دولي       0.80      0.80      0.80       700
       رياضة       0.98      0.99      0.98       700
       سياسة       0.78      0.79      0.79       700
       مجتمع       0.76      0.77      0.76       700

    accuracy                           0.84      4200
   macro avg       0.84      0.84      0.84      4200
weighted avg       0.84      0.84      0.84      4200


===== CONFUSION MATRIX =====
Rows = actual labels; columns = predicted labels


,اقتصاد,ثقافة,دولي,رياضة,سياسة,مجتمع
اقتصاد,552,23,29,1,44,51
ثقافة,17,635,13,3,15,17
دولي,21,14,560,2,54,49
رياضة,3,1,1,690,3,2
سياسة,32,13,45,3,556,51
مجتمع,51,18,51,3,41,536


In [36]:
import joblib

joblib.dump(final_model_ar, "embedding_classifier_mlt.joblib")

['embedding_classifier_mlt.joblib']

In [37]:
from huggingface_hub import login

login()

In [38]:
from huggingface_hub import HfApi

api = HfApi()

api.upload_file(
    path_or_fileobj="embedding_classifier_mlt.joblib",
    path_in_repo="embedding_classifier_mlt.joblib",
    repo_id="salah-2005/multilingual_embeddings",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ing_classifier_mlt.joblib: 100%|##########| 19.5kB / 19.5kB            

CommitInfo(commit_url='https://huggingface.co/salah-2005/multilingual_embeddings/commit/a21505b7beb20d492c57fe58e380ad0ea25233f0', commit_message='Upload embedding_classifier_mlt.joblib with huggingface_hub', commit_description='', oid='a21505b7beb20d492c57fe58e380ad0ea25233f0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/salah-2005/multilingual_embeddings', endpoint='https://huggingface.co', repo_type='model', repo_id='salah-2005/multilingual_embeddings'), pr_revision=None, pr_num=None)

In [39]:
import joblib
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

repo_id = "salah-2005/multilingual_embeddings"

classifier_path = hf_hub_download(
    repo_id=repo_id,
    filename="embedding_classifier_mlt.joblib"
)

classifier = joblib.load(classifier_path)

embedding_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_classifier_mlt.joblib: reconstructing file:   0%|          |  0.00B / 19.5kB            

embedding_classifier_mlt.joblib: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]